In [ ]:
# @title Install required dependencies
!pip install -q "crewai[tools]>=1.8.0" python-dotenv ddgs pydantic pandas pypdf


In [ ]:
import os
import csv
from pydantic import BaseModel
from google.colab import userdata
from crewai import Agent, Task, Crew, LLM
from crewai.flow.flow import Flow, listen, router, start

SAMPLE_RESUME = """
Candidate: Jordan Lee
Experience: 6 years as Full Stack Engineer
Skills: Python, FastAPI, Docker, PostgreSQL, React, Kubernetes, LLM integration
Education: B.S. in Computer Science
Achievements: Built real-time streaming pipeline reducing latency by 45%; deployed production CrewAI agents.
"""

class ScreeningState(BaseModel):
    candidate: str = "Jordan Lee"
    resume_text: str = SAMPLE_RESUME
    score: int = 0
    decision: str = ""
    evaluation: str = ""

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    llm = LLM(
        model="google/gemini-3.1-flash-lite-preview",
        api_key=userdata.get("GEMINI_API_KEY"),
        temperature=0.3,
        max_tokens=1000,
    )

    class ResumeScreeningFlow(Flow[ScreeningState]):
        @start()
        def ingest_resume(self):
            print(f"Screening candidate: {self.state.candidate}")

        @listen(ingest_resume)
        def score_resume(self):
            prompt = (
                f"Evaluate this resume for a Senior AI/Backend Engineer position on a scale from 1 to 10.\n"
                f"Resume: {self.state.resume_text}\n"
                f"Reply with ONLY an integer between 1 and 10."
            )
            raw = llm.call(prompt).strip()
            digits = "".join([c for c in raw if c.isdigit()])
            self.state.score = int(digits[:2]) if digits else 5

        @router(score_resume)
        def evaluate_decision(self):
            if self.state.score >= 7:
                self.state.decision = "shortlist"
                return "shortlist"
            else:
                self.state.decision = "reject"
                return "reject"

        @listen("shortlist")
        def process_shortlist(self):
            interviewer = Agent(
                role="Technical Interview Lead",
                goal="Highlight candidate strengths, stack compatibility, and recommend interview focus areas",
                backstory="Engineering director assembling high-performing product engineering teams.",
                llm=llm,
                verbose=False,
            )
            task = Task(
                description=f"Draft technical interview summary for {self.state.candidate} based on resume: {self.state.resume_text}",
                expected_output="A shortlist recommendation memo with suggested interview topics.",
                agent=interviewer,
            )
            crew = Crew(agents=[interviewer], tasks=[task], verbose=False)
            res = crew.kickoff()
            self.state.evaluation = res.raw

        @listen("reject")
        def process_reject(self):
            prompt = f"Write a courteous, respectful candidate rejection email to {self.state.candidate} for a Software Engineer application."
            self.state.evaluation = llm.call(prompt)

    flow = ResumeScreeningFlow()
    flow.kickoff()
    print(f"\n--- Candidate: {flow.state.candidate} | Score: {flow.state.score}/10 | Outcome: {flow.state.decision.upper()} ---")
    print("\nEvaluation / Message:\n", flow.state.evaluation)

except Exception as e:
    print("Error:", e)
